# LeetCode 900 — RLE Iterator bằng Python

Link đề: https://leetcode.com/problems/rle-iterator/ · Độ khó: **Medium** · Chủ đề: Array, Design, Counting, Iterator

Skeleton LeetCode:

```python
class RLEIterator:

    def __init__(self, encoding: List[int]):
        ...

    def next(self, n: int) -> int:
        ...

# Your RLEIterator object will be instantiated and called as such:
# obj = RLEIterator(encoding)
# param_1 = obj.next(n)
```

Ghi chú: đây là bài **thiết kế lớp**. LeetCode chạy một **chuỗi lệnh** (`["RLEIterator","next",…]` kèm danh sách tham số); notebook tự dựng bộ chạy chuỗi lệnh `run_ops`. Điểm cốt lõi: độ dài mỗi đoạn tới `10^9` nên **không thể bung** dãy; ta tiêu thụ ngay trên dạng nén bằng **hai con trỏ** (đoạn hiện tại + số phần tử đã dùng trong đoạn). Oracle: **bung dãy thật** thành danh sách rồi dùng con trỏ (chỉ cho độ dài nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mã hoá **độ dài lặp** (RLE) `[c0, v0, c1, v1, …]` nghĩa là giá trị `v_k` lặp `c_k` lần; `next(n)` **tiêu thụ** `n` phần tử kế tiếp và trả về phần tử **cuối cùng** bị tiêu thụ, hoặc `-1` nếu không đủ `n` phần tử.
2. Nắm công cụ Python: thiết kế lớp có trạng thái, `bisect`, `itertools.accumulate`, bộ chạy chuỗi lệnh kiểu LeetCode (`getattr`).
3. Đi từ **bung dãy thành danh sách (brute force)** → **hai con trỏ trên dạng nén** → **sửa mảng tại chỗ** → **tổng tiền tố + nhị phân**.
4. Hiểu mấu chốt: trạng thái chỉ cần `(đoạn hiện tại, đã dùng bao nhiêu)`; con trỏ chỉ **tiến**, nên tổng chi phí mọi lệnh là `O(len(encoding) + số lệnh)`.
5. Tránh lỗi kinh điển: bung dãy (hết bộ nhớ), quên bỏ qua đoạn có độ dài `0`, trả giá trị đoạn **kế tiếp** khi vừa dùng hết một đoạn, quên đặt lại bộ đếm khi sang đoạn mới, trả phần tử cuối khi không đủ `n` thay vì `-1`, và dùng `or -1` làm sai giá trị `0`.

## 2. Đề bài & ví dụ

Mã hoá độ dài lặp (RLE) của một dãy số nguyên: mảng `encoding` độ dài chẵn, với mọi chỉ số chẵn `i`, `encoding[i]` là **số lần** giá trị không âm `encoding[i + 1]` được lặp.

Ví dụ dãy `[8,8,8,5,5]` có thể mã hoá `[3,8,2,5]`; cũng hợp lệ: `[3,8,0,9,2,5]` (có đoạn dài `0`) và `[2,8,1,8,2,5]`.

Thiết kế `RLEIterator`:

- `RLEIterator(int[] encoded)`: khởi tạo với mảng mã hoá.
- `int next(int n)`: **tiêu thụ** `n` phần tử kế tiếp và trả về phần tử **cuối cùng** bị tiêu thụ; nếu không còn đủ phần tử (ít hơn `n`), trả về `-1`.

**Ví dụ 1:** `encoding = [3,8,0,9,2,5]` (dãy `[8,8,8,5,5]`); `next(2) → 8`, `next(1) → 8`, `next(1) → 5`, `next(2) → -1` (chỉ còn `1` phần tử).

**Ràng buộc:** `2 <= encoding.length <= 1000` (chẵn); `0 <= encoding[i] <= 10^9`; `1 <= n <= 10^9`; tối đa `1000` lệnh `next`.

In [1]:
import itertools
import random
import time
from bisect import bisect_left
from typing import List


def run_ops(cls, ops, args):
    """Bộ chạy chuỗi lệnh kiểu LeetCode: ops[0] là tên lớp (khởi tạo), các lệnh sau gọi phương thức cùng tên.
    Trả về danh sách kết quả (None cho lệnh khởi tạo)."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


example_ops = ["RLEIterator", "next", "next", "next", "next"]
example_args = [[[3, 8, 0, 9, 2, 5]], [2], [1], [1], [2]]
example_out = [None, 8, 8, 5, -1]
print("ví dụ:", example_ops, example_args, "->", example_out)

ví dụ: ['RLEIterator', 'next', 'next', 'next', 'next'] [[[3, 8, 0, 9, 2, 5]], [2], [1], [1], [2]] -> [None, 8, 8, 5, -1]


## 3. Kiến thức nền

### 3.1 RLE: nén và bung

`rle_decode(enc)` nối `v` lặp `c` lần cho mỗi cặp `(c, v)`. Có **nhiều** mã hoá cho cùng một dãy (đoạn độ dài `0`, đoạn bị tách).

In [2]:
def rle_decode(enc):
    out = []
    for c, v in zip(enc[0::2], enc[1::2]):
        out.extend([v] * c)
    return out


def rle_encode(seq):
    out = []
    for v, grp in itertools.groupby(seq):
        out += [len(list(grp)), v]
    return out


seq = [8, 8, 8, 5, 5]
print(rle_encode(seq), rle_decode([3, 8, 2, 5]) == rle_decode([3, 8, 0, 9, 2, 5]) == rle_decode([2, 8, 1, 8, 2, 5]) == seq)
print("giới hạn: một cặp [10**9, 7] bung ra", 10 ** 9, "phần tử ~", 8 * 10 ** 9 // 10 ** 9, "GB (danh sách int)")

[3, 8, 2, 5] True
giới hạn: một cặp [10**9, 7] bung ra 1000000000 phần tử ~ 8 GB (danh sách int)


### 3.2 Lớp, trạng thái và "chuỗi lệnh"

Bài thiết kế lớp: LeetCode tạo một đối tượng rồi gọi phương thức theo danh sách. Trạng thái của đối tượng phải đủ để trả lời mọi lệnh sau. Ở đây trạng thái là **vị trí hiện tại trong dạng nén**.

### 3.3 Con trỏ chỉ tiến → chi phí gộp (amortized)

Mỗi lệnh có thể "nhảy qua" nhiều đoạn, nhưng con trỏ đoạn **không bao giờ lùi**, nên tổng số lần nhảy qua mọi lệnh `≤ số đoạn`. Tổng chi phí `O(số đoạn + số lệnh)`.

### 3.4 Tổng tiền tố + nhị phân

Với `prefix[k] = c_0 + … + c_k`, phần tử thứ `p` (đếm từ 1) của dãy thuộc đoạn nhỏ nhất `k` có `prefix[k] ≥ p`: `bisect_left(prefix, p)`.

In [3]:
from itertools import accumulate
enc = [3, 8, 0, 9, 2, 5]
counts, vals = enc[0::2], enc[1::2]
prefix = list(accumulate(counts))
print("prefix =", prefix, "-> phần tử thứ p thuộc đoạn:", {p: bisect_left(prefix, p) for p in range(1, 6)}, "giá trị:", {p: vals[bisect_left(prefix, p)] for p in range(1, 6)})

prefix = [3, 3, 5] -> phần tử thứ p thuộc đoạn: {1: 0, 2: 0, 3: 0, 4: 2, 5: 2} giá trị: {1: 8, 2: 8, 3: 8, 4: 5, 5: 5}


## 4. Ý tưởng nền tảng / trực giác

`encoding = [3,8,0,9,2,5]` ⇒ dãy `[8,8,8,5,5]` (đoạn `0×9` không có phần tử).

```
đoạn:   [ 3 × 8 ] [ 0 × 9 ] [ 2 × 5 ]
dãy:     8  8  8     (rỗng)    5  5
```

Trạng thái: "đang ở đoạn `i`, đã dùng `j` phần tử của nó" — ban đầu `(0, 0)`.

- `next(2)`: còn trong đoạn `0` có `3 − 0 = 3 ≥ 2` ⇒ dùng `2`, `j = 2`, trả `8`.
- `next(1)`: còn `1 ≥ 1` ⇒ `j = 3`, trả `8` (đoạn `0` vừa hết).
- `next(1)`: còn `0 < 1` ⇒ nhảy sang đoạn `1` (độ dài `0`, bỏ qua), rồi đoạn `2`: dùng `1`, trả `5`.
- `next(2)`: còn `1 < 2` ⇒ hết dãy ⇒ `-1`.

In [4]:
flat = rle_decode([3, 8, 0, 9, 2, 5])
print('dãy bung:', flat, '| next(2), next(1), next(1), next(2) mong đợi:', example_out[1:])

dãy bung: [8, 8, 8, 5, 5] | next(2), next(1), next(1), next(2) mong đợi: [8, 8, 5, -1]


## 5. Cách 1 — Brute Force: bung dãy thành danh sách

**Ý tưởng:** ở hàm khởi tạo bung toàn bộ dãy `flat` (`rle_decode`), giữ chỉ số `pos` của phần tử kế tiếp. `next(n)`: nếu `pos + n > len(flat)` thì đặt `pos = len(flat)` và trả `-1`; ngược lại `pos += n` và trả `flat[pos − 1]`.

In [5]:
class RLEFlat:
    def __init__(self, encoding):
        self.flat = rle_decode(encoding)                 # bung toàn bộ: chỉ dùng được khi tổng độ dài nhỏ
        self.pos = 0

    def next(self, n):
        if self.pos + n > len(self.flat):                # không đủ n phần tử: tiêu thụ hết, trả -1
            self.pos = len(self.flat)
            return -1
        self.pos += n
        return self.flat[self.pos - 1]


assert run_ops(RLEFlat, example_ops, example_args) == example_out
print(run_ops(RLEFlat, example_ops, example_args))

[None, 8, 8, 5, -1]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng hết bộ nhớ khi độ dài lặp tới 10^9)
class RLEIteratorFlat:

    def __init__(self, encoding: List[int]):
        self.flat = []
        for c, v in zip(encoding[0::2], encoding[1::2]):
            self.flat.extend([v] * c)
        self.pos = 0

    def next(self, n: int) -> int:
        if self.pos + n > len(self.flat):
            self.pos = len(self.flat)
            return -1
        self.pos += n
        return self.flat[self.pos - 1]

**Độ phức tạp:** khởi tạo `O(Σ c_k)` thời gian và bộ nhớ (tới `5·10^11` phần tử!), `next` `O(1)`.

**Nút thắt:** ta lưu từng phần tử trong khi chỉ cần biết **ranh giới các đoạn**.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ nguyên dạng nén. Trạng thái `(i, used)`: đoạn `i` và số phần tử đã tiêu thụ trong đoạn đó. `next(n)`:

1. Trong khi còn đoạn và `n > c_i − used`: `n −= (c_i − used)`, sang đoạn kế (`i += 1`, `used = 0`).
2. Nếu hết đoạn: trả `-1`. Ngược lại `used += n`, trả `v_i`.

Mỗi vòng `while` làm `i` tăng, nên tổng chi phí gộp `O(số đoạn + số lệnh)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** hai con trỏ `(i, used)`, **Cách 3** sửa mảng tại chỗ, **Cách 4** tổng tiền tố + `bisect`.

### Cách 2 — Hai con trỏ `(i, used)`

**Giải thích từng đoạn:**

1. `self.i` là chỉ số đoạn (bước 2 trên mảng phẳng), `self.used` là số phần tử đã tiêu thụ trong đoạn đó.
2. `next(n)`: `while i < len(encoding) and n > encoding[i] − used`: `n −= encoding[i] − used`; `used = 0`; `i += 2`.
3. Nếu `i ≥ len(encoding)`: trả `-1`. Ngược lại `used += n`, trả `encoding[i + 1]`.

In [7]:
class RLEPointers:
    def __init__(self, encoding):
        self.enc = encoding
        self.i = 0                                       # chỉ số của count của đoạn hiện tại
        self.used = 0                                    # số phần tử đã tiêu thụ trong đoạn hiện tại

    def next(self, n):
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used            # tiêu thụ hết phần còn lại của đoạn
            self.used = 0
            self.i += 2                                  # sang đoạn kế tiếp (đoạn dài 0 bị bỏ qua nhờ vòng lặp)
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]


assert run_ops(RLEPointers, example_ops, example_args) == example_out
print(run_ops(RLEPointers, example_ops, example_args))

[None, 8, 8, 5, -1]


In [8]:
# Code nộp LeetCode cho Cách 2
class RLEIteratorPointers:

    def __init__(self, encoding: List[int]):
        self.enc = encoding
        self.i = 0
        self.used = 0

    def next(self, n: int) -> int:
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]

**Độ phức tạp:** khởi tạo `O(1)`; `next` `O(1)` gộp (tổng `O(số đoạn + số lệnh)`); bộ nhớ `O(1)` thêm.

### Cách 3 — Sửa mảng tại chỗ

**Ý tưởng:** thay vì nhớ `used`, **trừ trực tiếp** vào `encoding[i]`: mỗi lần tiêu thụ `m` phần tử của đoạn `i`, `encoding[i] -= m`. Khi `encoding[i]` về `0` thì đoạn đó hết, tăng `i`. Con trỏ `i` vẫn chỉ tiến.

In [9]:
class RLEMutate:
    def __init__(self, encoding):
        self.enc = list(encoding)                        # sao chép: không phá mảng của người gọi
        self.i = 0

    def next(self, n):
        while self.i < len(self.enc) and self.enc[self.i] < n:
            n -= self.enc[self.i]                        # đoạn không đủ: dùng hết và bỏ
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.enc[self.i] -= n                            # tiêu thụ n phần tử của đoạn này
        return self.enc[self.i + 1]


assert run_ops(RLEMutate, example_ops, example_args) == example_out
print(run_ops(RLEMutate, example_ops, example_args))

[None, 8, 8, 5, -1]


In [10]:
# Code nộp LeetCode cho Cách 3
class RLEIteratorMutate:

    def __init__(self, encoding: List[int]):
        self.enc = encoding
        self.i = 0

    def next(self, n: int) -> int:
        while self.i < len(self.enc) and self.enc[self.i] < n:
            n -= self.enc[self.i]
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.enc[self.i] -= n
        return self.enc[self.i + 1]

**Độ phức tạp:** như Cách 2. Ưu điểm: chỉ một con trỏ. Nhược điểm: làm hỏng mảng đầu vào (nên sao chép nếu cần).

### Cách 4 — Tổng tiền tố + `bisect`

**Ý tưởng:** `prefix[k]` = tổng độ dài `k + 1` đoạn đầu; `total = prefix[-1]`. Giữ `consumed` = số phần tử đã tiêu thụ. `next(n)`: nếu `consumed + n > total` thì `consumed = total`, trả `-1`; ngược lại `consumed += n`, đoạn chứa phần tử thứ `consumed` là `bisect_left(prefix, consumed)`; trả giá trị đoạn đó. Mỗi `next` là `O(log số đoạn)` (không cần con trỏ trạng thái).

In [11]:
class RLEPrefix:
    def __init__(self, encoding):
        self.vals = encoding[1::2]
        self.prefix = list(itertools.accumulate(encoding[0::2]))     # prefix[k] = tổng độ dài các đoạn 0..k
        self.total = self.prefix[-1]
        self.consumed = 0

    def next(self, n):
        if self.consumed + n > self.total:
            self.consumed = self.total
            return -1
        self.consumed += n
        k = bisect_left(self.prefix, self.consumed)      # đoạn đầu tiên có prefix >= consumed (đoạn không rỗng)
        return self.vals[k]


assert run_ops(RLEPrefix, example_ops, example_args) == example_out
print(run_ops(RLEPrefix, example_ops, example_args))

[None, 8, 8, 5, -1]


In [12]:
# Code nộp LeetCode cho Cách 4
from bisect import bisect_left
from itertools import accumulate


class RLEIteratorPrefix:

    def __init__(self, encoding: List[int]):
        self.vals = encoding[1::2]
        self.prefix = list(accumulate(encoding[0::2]))
        self.consumed = 0

    def next(self, n: int) -> int:
        if self.consumed + n > self.prefix[-1]:
            self.consumed = self.prefix[-1]
            return -1
        self.consumed += n
        return self.vals[bisect_left(self.prefix, self.consumed)]

**Độ phức tạp:** khởi tạo `O(số đoạn)`; `next` `O(log số đoạn)`; bộ nhớ `O(số đoạn)`. Đoạn có độ dài `0` không bao giờ được chọn: nếu `c_k = 0` thì `prefix[k] = prefix[k−1]`, nên `bisect_left` (vị trí đầu tiên có `prefix ≥ consumed`) dừng ở đoạn **trước** đó hoặc sớm hơn.

## 8. TRACE từng bước

**Trace Cách 2** với `encoding = [3,8,0,9,2,5]`:

| lệnh | trạng thái trước `(i, used)` | vòng `while` | trạng thái sau | trả về |
|---|---|---|---|---|
| `next(2)` | `(0, 0)` | `2 > 3 − 0`? không | `(0, 2)` | `enc[1] = 8` |
| `next(1)` | `(0, 2)` | `1 > 3 − 2`? không | `(0, 3)` | `8` |
| `next(1)` | `(0, 3)` | `1 > 3 − 3 = 0`: có → `n = 1 − 0`, `used = 0`, `i = 2`; `1 > 0 − 0`: có → `n = 1`, `i = 4`; `1 > 2 − 0`? không | `(4, 1)` | `enc[5] = 5` |
| `next(2)` | `(4, 1)` | `2 > 2 − 1 = 1`: có → `n = 1`, `i = 6` (hết) | `(6, 0)` | `-1` |

Hàm trace in từng bước:

In [13]:
def trace_pointers(encoding, calls):
    i, used = 0, 0
    print(f"encoding = {encoding}")
    print(f"{'next(n)':>8} {'trước (i,used)':>15}  các lần nhảy đoạn                       {'sau':>9} trả về")
    for n in calls:
        before = (i, used)
        jumps = []
        n0 = n
        while i < len(encoding) and n > encoding[i] - used:
            jumps.append(f"bỏ đoạn {i // 2} (còn {encoding[i] - used})")
            n -= encoding[i] - used
            used = 0
            i += 2
        if i >= len(encoding):
            res = -1
        else:
            used += n
            res = encoding[i + 1]
        print(f"{'next(' + str(n0) + ')':>8} {str(before):>15}  {'; '.join(jumps) or '(không nhảy)':<38} {str((i, used)):>9} {res}")
    return


trace_pointers([3, 8, 0, 9, 2, 5], [2, 1, 1, 2])

encoding = [3, 8, 0, 9, 2, 5]
 next(n)  trước (i,used)  các lần nhảy đoạn                             sau trả về
 next(2)          (0, 0)  (không nhảy)                              (0, 2) 8
 next(1)          (0, 2)  (không nhảy)                              (0, 3) 8
 next(1)          (0, 3)  bỏ đoạn 0 (còn 0); bỏ đoạn 1 (còn 0)      (4, 1) 5
 next(2)          (4, 1)  bỏ đoạn 2 (còn 1)                         (6, 0) -1


**Trace Cách 4:** `prefix = [3, 3, 5]`, `total = 5`. `next(2)`: `consumed = 2`, `bisect_left(prefix, 2) = 0` → giá trị `8`. `next(1)`: `consumed = 3` → `bisect_left = 0` (`prefix[0] = 3 ≥ 3`) → `8`. `next(1)`: `consumed = 4` → `bisect_left = 2` → `5`. `next(2)`: `4 + 2 > 5` ⇒ `-1`.

## 9. Code nộp LeetCode

Chọn **Cách 2** (hai con trỏ). Giữ nguyên tên lớp `RLEIterator`, hàm `next(self, n)`:

In [14]:
class RLEIterator:

    def __init__(self, encoding: List[int]):
        self.enc = encoding
        self.i = 0                                        # chỉ số count của đoạn hiện tại
        self.used = 0                                     # đã dùng bao nhiêu phần tử của đoạn hiện tại

    def next(self, n: int) -> int:
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used             # dùng hết phần còn lại của đoạn rồi sang đoạn sau
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):                       # hết dãy mà vẫn thiếu phần tử
            return -1
        self.used += n
        return self.enc[self.i + 1]


print(run_ops(RLEIterator, example_ops, example_args))      # [None, 8, 8, 5, -1]
assert run_ops(RLEIterator, example_ops, example_args) == example_out

[None, 8, 8, 5, -1]


## 10. Vì sao đúng

**(a) Mô hình.** Dãy là nối các khối `v_k` lặp `c_k` lần. `next(n)` tiêu thụ `n` phần tử liên tiếp bắt đầu từ vị trí con trỏ ảo `pos`; trả `seq[pos + n − 1]` nếu `pos + n ≤ len(seq)`, ngược lại `-1` (và dãy coi như hết, `pos = len(seq)`).

**(b) Bất biến của Cách 2.** `(i, used)` biểu diễn `pos = Σ_{k < i/2} c_k + used` với `0 ≤ used ≤ c_{i/2}` khi `i` còn trong mảng. `next(n)`:

- Vòng `while`: nếu `n > c − used` (còn lại của đoạn hiện tại) thì phần tử thứ `n` nằm **sau** đoạn này; trừ `c − used` khỏi `n` và nhảy sang đoạn kế với `used = 0`. Đoạn có `c = 0` thoả `n > 0 − 0` (vì `n ≥ 1`) nên bị bỏ qua.
- Khi thoát vòng với `i` còn hợp lệ: `1 ≤ n ≤ c − used`, nên phần tử thứ `n` tính từ vị trí hiện tại nằm đúng trong đoạn `i` (giá trị `v`), và `used += n` giữ bất biến.
- Nếu `i ≥ len(enc)`: số phần tử còn lại ít hơn `n` ⇒ `-1`; `i` đã ở cuối nên các lệnh sau cũng cho `-1` (dãy hết).

**(c) `n ≥ 1`.** Vì `n ≥ 1`, sau vòng `while` `n ≥ 1` luôn (ta chỉ trừ khi `n > còn lại ≥ 0`), do đó trả đúng "phần tử thứ `n`" chứ không phải phần tử trước vị trí hiện tại.

**(d) Chi phí gộp.** `i` chỉ tăng, mỗi lần tăng gắn với một vòng `while`; tổng vòng `≤ số đoạn`; cộng `O(1)` mỗi lệnh ⇒ `O(số đoạn + số lệnh)`.

**(e) Cách 3** là cùng ý với `used` được "hấp thụ" vào `enc[i]`. **Cách 4**: phần tử thứ `p` thuộc đoạn nhỏ nhất `k` có `prefix[k] ≥ p`; đoạn rỗng không bao giờ là nhỏ nhất như vậy.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **bung dãy thật** trên mọi mã hoá nhỏ và mọi chuỗi lệnh ngắn (mục 12D), và khớp nhau với độ dài tới `10^9` (12F).

## 11. Time / Space Complexity

| Cách | Khởi tạo | `next` | Space | Ghi chú |
|---|---|---|---|---|
| 1. Bung dãy | `O(Σ c)` | `O(1)` | `O(Σ c)` | oracle, độ dài nhỏ |
| 2. Hai con trỏ | `O(1)` | `O(1)` gộp | `O(1)` | **đáp án chính** |
| 3. Sửa tại chỗ | `O(1)` | `O(1)` gộp | `O(1)` | phá mảng đầu vào |
| 4. Tiền tố + `bisect` | `O(m)` | `O(log m)` | `O(m)` | không con trỏ trạng thái |

Đo thực tế (`m = 500` đoạn, độ dài tới `10^9`, 1000 lệnh):

In [15]:
def bench_run(cls, enc, calls, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        it = cls(list(enc))
        for n in calls:
            it.next(n)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(900)
enc_big = []
for _ in range(500):
    enc_big += [rng.randint(0, 10 ** 9), rng.randint(0, 10 ** 9)]
calls_big = [rng.randint(1, 10 ** 9) for _ in range(1000)]
for name, cls in [("pointers", RLEPointers), ("mutate", RLEMutate), ("prefix", RLEPrefix)]:
    print(f"{name:9s}: {bench_run(cls, enc_big, calls_big) * 1e3:7.3f} ms cho khởi tạo + 1000 lệnh")
small = [3, 8, 0, 9, 2, 5] * 5
print(f"flat với tổng độ dài nhỏ: {bench_run(RLEFlat, [rng.randint(0, 50) if k % 2 == 0 else k for k in range(100)], [1, 2, 3] * 100) * 1e3:7.3f} ms (không dùng được khi độ dài lớn)")

pointers :   0.226 ms cho khởi tạo + 1000 lệnh
mutate   :   0.192 ms cho khởi tạo + 1000 lệnh
prefix   :   0.165 ms cho khởi tạo + 1000 lệnh
flat với tổng độ dài nhỏ:   0.041 ms (không dùng được khi độ dài lớn)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề (dùng `run_ops`), (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mã hoá nhỏ và mọi chuỗi lệnh ngắn với oracle bung dãy, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`10^9`).

### A. Ví dụ của đề

In [16]:
class RLEOracle:
    """Oracle: bung dãy thật bằng itertools và dùng một iterator duy nhất; next(n) lấy n phần tử bằng islice."""
    def __init__(self, encoding):
        self.it = itertools.chain.from_iterable(itertools.repeat(v, c) for c, v in zip(encoding[0::2], encoding[1::2]))

    def next(self, n):
        got = list(itertools.islice(self.it, n))          # tiêu thụ tối đa n phần tử
        return got[-1] if len(got) == n else -1


CLASSES = {
    "flat": RLEFlat,
    "pointers": RLEPointers,
    "mutate": RLEMutate,
    "prefix": RLEPrefix,
    "RLEIterator": RLEIterator,
    "RLEIteratorFlat": RLEIteratorFlat,
    "RLEIteratorPointers": RLEIteratorPointers,
    "RLEIteratorMutate": RLEIteratorMutate,
    "RLEIteratorPrefix": RLEIteratorPrefix,
}


def run_all(enc, calls, names=None):
    oracle = RLEOracle(list(enc))
    want = [oracle.next(n) for n in calls]
    for name in (names or CLASSES):
        it = CLASSES[name](list(enc))
        got = [it.next(n) for n in calls]
        assert got == want, (name, enc, calls, got, want)
    return want


for cls in CLASSES.values():
    assert run_ops(cls, example_ops, [[list(example_args[0][0])]] + example_args[1:]) == example_out
assert run_all([3, 8, 0, 9, 2, 5], [2, 1, 1, 2]) == [8, 8, 5, -1]
print("ví dụ đề: oracle bung dãy và", len(CLASSES), "cài đặt (chạy bằng run_ops) đều cho", example_out)

ví dụ đề: oracle bung dãy và 9 cài đặt (chạy bằng run_ops) đều cho [None, 8, 8, 5, -1]


### B. Edge case

- Mọi độ dài `0` ⇒ mọi `next` cho `-1`.
- `next(n)` đúng bằng số phần tử còn lại; `next(n)` thiếu **một** phần tử.
- Sau khi trả `-1`, lệnh tiếp theo cũng `-1`.
- Đoạn độ dài `0` ở đầu, giữa, cuối.
- Hai đoạn liền nhau cùng giá trị.
- Giá trị `0` (không nhầm với `-1` hay "sai").
- `next` bằng đúng bội ranh giới đoạn.

In [17]:
cases = [
    ([0, 5, 0, 7], [1, 1], [-1, -1]),
    ([2, 9], [2], [9]), ([2, 9], [3], [-1]), ([2, 9], [1, 1, 1], [9, 9, -1]),
    ([0, 1, 3, 4, 0, 2], [1, 1, 1, 1], [4, 4, 4, -1]),
    ([1, 1, 1, 1, 1, 2], [1, 1, 1, 1], [1, 1, 2, -1]),
    ([2, 0, 2, 0], [1, 1, 1, 1, 1], [0, 0, 0, 0, -1]),
    ([3, 8, 2, 5], [3, 2, 1], [8, 5, -1]), ([3, 8, 2, 5], [5], [5]), ([3, 8, 2, 5], [6], [-1]),
    ([1, 7], [1, 1, 1], [7, -1, -1]),
]
for enc, calls, want in cases:
    assert run_all(enc, calls) == want, (enc, calls)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

11 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
class BugNextRunValue:                                  # khi dùng hết đoạn thì trả giá trị đoạn KẾ TIẾP
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        while self.i < len(self.enc) and n >= self.enc[self.i] - self.used:      # SAI: >= thay vì >
            n -= self.enc[self.i] - self.used
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]


class BugNoResetUsed:                                   # quên used = 0 khi sang đoạn mới
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used
            self.i += 2                                  # SAI: không đặt lại used
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]


class BugLastOnPartial:                                 # không đủ n phần tử mà vẫn trả giá trị cuối cùng
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        last = -1
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            if self.enc[self.i] - self.used > 0:
                last = self.enc[self.i + 1]
            n -= self.enc[self.i] - self.used
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):
            return last                                  # SAI: phải là -1
        self.used += n
        return self.enc[self.i + 1]


class BugOrMinusOne(RLEPointers):                       # dùng `or -1`: giá trị 0 bị biến thành -1
    def next(self, n):
        r = super().next(n)
        return r or -1


class BugNoResetAfterFail(RLEPrefix):                   # không chặn consumed khi thất bại -> lệnh sau vẫn có thể trả giá trị
    def next(self, n):
        if self.consumed + n > self.total:
            return -1                                    # SAI: không đặt consumed = total
        return super().next(n)


BUGS = [BugNextRunValue, BugNoResetUsed, BugLastOnPartial, BugOrMinusOne, BugNoResetAfterFail]
rng = random.Random(3)
tests = []
for _ in range(1500):
    enc = []
    for _ in range(rng.randint(1, 4)):
        enc += [rng.randint(0, 4), rng.randint(0, 3)]
    tests.append((enc, [rng.randint(1, 4) for _ in range(rng.randint(1, 6))]))
for bug in BUGS:
    bad = None
    for enc, calls in tests:
        o = RLEOracle(list(enc))
        want = [o.next(n) for n in calls]
        it = bug(list(enc))
        got = [it.next(n) for n in calls]
        if got != want:
            bad = (enc, calls, got, want)
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai: encoding = {bad[0]}, calls = {bad[1]}: ra {bad[2]}, đúng {bad[3]}")

BugNextRunValue      sai: encoding = [4, 1, 2, 3], calls = [1, 1, 4, 3, 2, 2]: ra [1, 1, -1, -1, -1, -1], đúng [1, 1, 3, -1, -1, -1]
BugNoResetUsed       sai: encoding = [4, 1, 2, 3], calls = [1, 1, 4, 3, 2, 2]: ra [1, 1, -1, -1, -1, -1], đúng [1, 1, 3, -1, -1, -1]
BugLastOnPartial     sai: encoding = [1, 3], calls = [3, 4]: ra [3, -1], đúng [-1, -1]
BugOrMinusOne        sai: encoding = [4, 0, 1, 2, 2, 0], calls = [4]: ra [-1], đúng [0]
BugNoResetAfterFail  sai: encoding = [1, 0, 2, 2, 2, 1], calls = [4, 4, 4, 1]: ra [1, -1, -1, 1], đúng [1, -1, -1, -1]


### D. Vét cạn mọi mã hoá nhỏ và mọi chuỗi lệnh ngắn

Mọi `encoding` có `1..3` cặp, độ dài lặp `0..3`, giá trị `0..2` (`12 + 144 + 1 728 = 1 884` mã hoá) × mọi chuỗi lệnh `next` có độ dài `1..4` với `n ∈ {1, 2, 3}` (`3 + 9 + 27 + 81 = 120` chuỗi). Oracle: bung dãy.

In [19]:
t0 = time.perf_counter()
call_seqs = [list(s) for L in range(1, 5) for s in itertools.product((1, 2, 3), repeat=L)]
assert len(call_seqs) == 120
cnt = 0
fast = ("flat", "pointers", "mutate", "prefix", "RLEIterator")
for pairs in range(1, 4):
    for combo in itertools.product(itertools.product(range(4), range(3)), repeat=pairs):
        enc = [x for c, v in combo for x in (c, v)]
        for calls in call_seqs:
            run_all(enc, calls, names=fast)
            cnt += 1
print(cnt, "bộ (mã hoá, chuỗi lệnh): 5 cài đặt khớp oracle bung dãy;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 1884 * 120

226080 bộ (mã hoá, chuỗi lệnh): 5 cài đặt khớp oracle bung dãy; 2.0 s


### E. Ngẫu nhiên có seed và tính chất

Mã hoá ngẫu nhiên tới `6` cặp, độ dài lặp tới `6`, chuỗi `10` lệnh với `n ∈ [1, 8]`. Tính chất: (1) một khi trả `-1` thì mọi lệnh sau cũng `-1`; (2) tổng các `n` **thành công** (không `-1`) không vượt tổng độ dài; (3) kết quả khác `-1` luôn là một giá trị có mặt trong mã hoá (với đoạn độ dài `> 0`).

In [20]:
rng = random.Random(20240900)
for _ in range(3000):
    enc = []
    for _ in range(rng.randint(1, 6)):
        enc += [rng.randint(0, 6), rng.randint(0, 9)]
    calls = [rng.randint(1, 8) for _ in range(10)]
    want = run_all(enc, calls)
    total = sum(enc[0::2])
    if -1 in want:
        k = want.index(-1)
        assert all(x == -1 for x in want[k:])                                    # (1)
    ok_sum = sum(n for n, r in zip(calls, want) if r != -1)
    assert ok_sum <= total                                                        # (2)
    live_vals = {v for c, v in zip(enc[0::2], enc[1::2]) if c > 0}
    assert all(r in live_vals for r in want if r != -1)                           # (3)
print("3000 ca ngẫu nhiên: 9 cài đặt khớp oracle bung dãy; 3 tính chất đúng")

3000 ca ngẫu nhiên: 9 cài đặt khớp oracle bung dãy; 3 tính chất đúng


### F. Giới hạn: độ dài lặp tới `10^9`

Không bung được. So 3 cài đặt nhanh với nhau trên 1000 lệnh với `n` tới `10^9`; kiểm tra bảo toàn: `tổng n thành công ≤ tổng độ dài`; và các trường hợp biên `n = 10^9`, đoạn dài đúng `10^9`.

In [21]:
rng = random.Random(500)
cnt = 0
for _ in range(200):
    m = rng.randint(1, 500)
    enc = []
    for _ in range(m):
        enc += [rng.choice([0, rng.randint(0, 10 ** 9), 10 ** 9, rng.randint(1, 5)]), rng.randint(0, 10 ** 9)]
    calls = [rng.choice([1, rng.randint(1, 10 ** 9), 10 ** 9, rng.randint(1, 50)]) for _ in range(1000)]
    results = {}
    for name in ("pointers", "mutate", "prefix", "RLEIterator", "RLEIteratorMutate", "RLEIteratorPrefix"):
        it = CLASSES[name](list(enc))
        results[name] = [it.next(n) for n in calls]
    ref = results["pointers"]
    assert all(r == ref for r in results.values()), "các cài đặt nhanh phải khớp nhau"
    assert sum(n for n, r in zip(calls, ref) if r != -1) <= sum(enc[0::2])
    cnt += 1
print(cnt, "mã hoá lớn × 1000 lệnh: 6 cài đặt nhanh khớp nhau; bảo toàn tổng độ dài")
it = RLEIterator([10 ** 9, 7, 10 ** 9, 8])
assert [it.next(10 ** 9), it.next(1), it.next(10 ** 9 - 1), it.next(1), it.next(1)] == [7, 8, 8, -1, -1]
it = RLEIterator([10 ** 9, 7])
assert it.next(10 ** 9) == 7 and it.next(1) == -1
print("biên 10^9: đúng")

200 mã hoá lớn × 1000 lệnh: 6 cài đặt nhanh khớp nhau; bảo toàn tổng độ dài
biên 10^9: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng hết đoạn thì trả giá trị đoạn kế tiếp

Dùng `>=` thay vì `>` trong vòng `while` khiến con trỏ nhảy khỏi đoạn khi vừa vặn dùng hết nó.

In [22]:
it = BugNextRunValue([3, 8, 2, 5])
print("encoding [3,8,2,5], next(3):", it.next(3), "| đúng: 8")
assert BugNextRunValue([3, 8, 2, 5]).next(3) == 5 and RLEIterator([3, 8, 2, 5]).next(3) == 8

encoding [3,8,2,5], next(3): 5 | đúng: 8


**Vì sao sai:** `n = 3` đúng bằng độ dài đoạn đầu: phần tử thứ `3` thuộc đoạn đầu (`8`). **Đúng:** chỉ nhảy khi `n > còn lại`.

### Lỗi 2: quên đặt lại `used` khi sang đoạn mới

In [23]:
bad = next((enc, calls) for enc, calls in tests if (lambda it: [it.next(n) for n in calls])(BugNoResetUsed(list(enc))) != run_all(enc, calls, names=["pointers"]))
print(bad[0], bad[1], "->", (lambda it: [it.next(n) for n in bad[1]])(BugNoResetUsed(list(bad[0]))), "| đúng:", run_all(*bad, names=["pointers"]))

[4, 1, 2, 3] [1, 1, 4, 3, 2, 2] -> [1, 1, -1, -1, -1, -1] | đúng: [1, 1, 3, -1, -1, -1]


**Vì sao sai:** `used` đếm phần tử đã dùng **trong đoạn hiện tại**; sang đoạn mới phải về `0`. **Đúng:** `self.used = 0` khi nhảy.

### Lỗi 3: không đủ `n` phần tử mà vẫn trả giá trị cuối

In [24]:
print("[2,9], next(3):", BugLastOnPartial([2, 9]).next(3), "| đúng:", RLEIterator([2, 9]).next(3))
assert BugLastOnPartial([2, 9]).next(3) == 9 and RLEIterator([2, 9]).next(3) == -1

[2,9], next(3): 9 | đúng: -1


**Vì sao sai:** đề: nếu phần tử cuối cùng bị tiêu thụ **không tồn tại** thì trả `-1`. **Đúng:** hết dãy ⇒ `-1`.

### Lỗi 4: `or -1` làm sai giá trị `0`

In [25]:
print("[2,0], next(1):", BugOrMinusOne([2, 0]).next(1), "| đúng:", RLEIterator([2, 0]).next(1))
assert BugOrMinusOne([2, 0]).next(1) == -1 and RLEIterator([2, 0]).next(1) == 0

[2,0], next(1): -1 | đúng: 0


**Vì sao sai:** `0` là giá trị hợp lệ nhưng "falsy" trong Python. **Đúng:** so sánh rõ ràng (`if i >= len(...)`), không dùng `or`.

### Lỗi 5: không chặn bộ đếm khi thất bại

Sau lệnh thất bại, bộ đếm phải coi như đã tiêu thụ hết.

In [26]:
bad = next((enc, calls) for enc, calls in tests
           if (lambda it: [it.next(n) for n in calls])(BugNoResetAfterFail(list(enc))) != run_all(enc, calls, names=["pointers"]))
print(bad[0], bad[1], "->", (lambda it: [it.next(n) for n in bad[1]])(BugNoResetAfterFail(list(bad[0]))), "| đúng:", run_all(*bad, names=["pointers"]))

[1, 0, 2, 2, 2, 1] [4, 4, 4, 1] -> [1, -1, -1, 1] | đúng: [1, -1, -1, -1]


**Vì sao sai:** sau `next(n)` thất bại, ví dụ còn `1` phần tử và `next(2) → -1`, phần tử cuối **đã bị tiêu thụ**; lệnh `next(1)` sau đó phải trả `-1`. **Đúng:** `consumed = total` khi thất bại.

### Lỗi 6 (bonus): bung dãy

In [27]:
total = 10 ** 9 * 500
print("tổng độ dài có thể tới", total, "phần tử -> bung dãy cần ~", total * 8 // 10 ** 12, "TB bộ nhớ: không thể")

tổng độ dài có thể tới 500000000000 phần tử -> bung dãy cần ~ 4 TB bộ nhớ: không thể


**Vì sao sai:** độ dài lặp tới `10^9` mỗi đoạn. **Đúng:** chỉ làm việc trên dạng nén.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "bộ lặp trên dãy **nén**/khổng lồ, hỏi phần tử thứ `k` hoặc tiêu thụ `n` phần tử" → **con trỏ trên dạng nén** (trạng thái `(đoạn, đã dùng)`), hoặc **tổng tiền tố + nhị phân** khi cần truy cập ngẫu nhiên.

**Mẹo:**

1. Không bung dãy; lưu ranh giới đoạn.
2. Con trỏ chỉ tiến ⇒ chi phí gộp tuyến tính.
3. Vòng `while` với điều kiện `n > còn lại` tự bỏ qua đoạn độ dài `0`.
4. Phân biệt "dùng hết đoạn" (`n == còn lại`, vẫn ở đoạn đó) với "vượt đoạn" (`n > còn lại`).
5. Cẩn thận `0` là giá trị hợp lệ; trả `-1` cho "không tồn tại".
6. Viết bộ chạy chuỗi lệnh để test thiết kế lớp.

**Họ bài cùng pattern:** Design Compressed String Iterator (604, Premium), Flatten 2D Vector (251), Zigzag Iterator (281), Peeking Iterator (284), Decode String (394), Design Browser History (1472).

## 15. Biến thể

1. **`peek()`:** xem phần tử kế tiếp mà không tiêu thụ.
2. **`hasNext()`:** còn phần tử không?
3. **`remaining()`:** số phần tử còn lại.
4. **`reset()`:** quay về đầu.
5. **Compressed String Iterator (604):** mã hoá dạng `"L1e2t1C1o1d1e1"`.

Cài biến thể 1–3 (lớp mở rộng) và đối chiếu oracle:

In [28]:
class RLEIteratorExt(RLEPointers):
    def _normalize(self):
        while self.i < len(self.enc) and self.enc[self.i] - self.used == 0:
            self.used = 0
            self.i += 2

    def peek(self):
        self._normalize()
        return -1 if self.i >= len(self.enc) else self.enc[self.i + 1]

    def hasNext(self):
        self._normalize()
        return self.i < len(self.enc)

    def remaining(self):
        if self.i >= len(self.enc):
            return 0
        return self.enc[self.i] - self.used + sum(self.enc[k] for k in range(self.i + 2, len(self.enc), 2))


rng = random.Random(15)
for _ in range(500):
    enc = []
    for _ in range(rng.randint(1, 4)):
        enc += [rng.randint(0, 5), rng.randint(0, 3)]
    ext, flat = RLEIteratorExt(list(enc)), rle_decode(enc)
    pos = 0
    for _ in range(8):
        assert ext.remaining() == len(flat) - pos
        assert ext.hasNext() == (pos < len(flat))
        assert ext.peek() == (flat[pos] if pos < len(flat) else -1)
        n = rng.randint(1, 4)
        r = ext.next(n)
        if pos + n > len(flat):
            pos = len(flat)
            assert r == -1
        else:
            pos += n
            assert r == flat[pos - 1]
print("peek / hasNext / remaining khớp danh sách bung trên 500 ca")

peek / hasNext / remaining khớp danh sách bung trên 500 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Khởi tạo | `next` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Bung dãy | danh sách phẳng + chỉ số | `O(Σ c)` | `O(1)` | `O(Σ c)` | oracle |
| 2. Hai con trỏ | `(i, used)` | `O(1)` | `O(1)` gộp | `O(1)` | **đáp án chính** |
| 3. Sửa tại chỗ | trừ trực tiếp vào `enc[i]` | `O(1)` | `O(1)` gộp | `O(1)` | một con trỏ |
| 4. Tiền tố + `bisect` | `consumed` + `bisect_left` | `O(m)` | `O(log m)` | `O(m)` | truy cập ngẫu nhiên |

Cả bốn khớp oracle bung dãy trên `1 884 × 120` bộ (mã hoá, chuỗi lệnh) và khớp nhau với độ dài `10^9` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(run_ops(RLEIterator, ["RLEIterator", "next", "next", "next"], [[[1, 4, 2, 6]], [1], [2], [1]]),
      run_ops(RLEIterator, ["RLEIterator", "next", "next"], [[[0, 1, 3, 2]], [3], [1]]))

[None, 4, 6, -1] [None, 2, -1]


### Bài 2 (tính tay)

`encoding = [2, 1, 0, 5, 3, 7]`. (a) Dãy bung ra? (b) Với các lệnh `next(1), next(2), next(3), next(1)` điền kết quả và trạng thái `(i, used)` sau mỗi lệnh. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai con trỏ:

In [30]:
class PointersTodo:
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            pass        # TODO 1: trừ phần còn lại của đoạn khỏi n, đặt used = 0, i += 2
        if self.i >= len(self.enc):
            return -1
        pass            # TODO 2: used += n và trả giá trị đoạn hiện tại


# Khi làm xong, bỏ comment để kiểm tra:
# assert run_ops(PointersTodo, example_ops, example_args) == example_out

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MyIterator` bằng **tổng tiền tố + `bisect_left`**:

In [31]:
class MyIterator:
    def __init__(self, encoding):
        pass            # TODO

    def next(self, n):
        pass            # TODO


# Khi làm xong:
# assert run_ops(MyIterator, example_ops, example_args) == example_out

### Bài 5 (mini challenge): `Compressed String Iterator` (604)

Mã hoá dạng `"L1e2t1C1o1d1e1"` (chữ cái theo sau bởi số lần, số lần có thể nhiều chữ số). Viết `StringIterator(compressed)` với `next()` trả ký tự kế tiếp (hoặc `' '` nếu hết) và `hasNext()`. Kiểm tra với bản bung.

In [32]:
class StringIterator:
    def __init__(self, compressed):
        pass            # TODO

    def next(self):
        pass            # TODO

    def hasNext(self):
        pass            # TODO


# Khi làm xong:
# s = StringIterator("L1e2t1C1o1d1e1"); assert "".join(s.next() for _ in range(8)) == "LeetCode" and s.next() == " " 

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: dãy bung của `[1,4,2,6]` là `[4,6,6]`; của `[0,1,3,2]` là `[2,2,2]`.
- Mức 2: `next(1)=4`, `next(2)=6`, `next(1)=-1`; `next(3)=2`, `next(1)=-1`.
- Mức 3: `[None, 4, 6, -1]` và `[None, 2, -1]`.

**Bài 2**

- Mức 1: đoạn `2×1`, `0×5` (rỗng), `3×7` → `[1,1,7,7,7]`.
- Mức 2: `next(1)=1`; `next(2)`: còn `1 < 2` → nhảy; được `7`; `next(3)`: còn `2 < 3` → `-1`.
- Mức 3: kết quả `1, 7, -1, -1`; trạng thái `(0,1)`, `(4,1)`, `(6,0)`, `(6,0)`.

**Bài 3**

- Mức 1: `n -= self.enc[self.i] - self.used`.
- Mức 2: `self.used = 0` và `self.i += 2`.
- Mức 3: sau vòng lặp `self.used += n; return self.enc[self.i + 1]`.

**Bài 4**

- Mức 1: `prefix = list(accumulate(enc[0::2]))`.
- Mức 2: `consumed + n > total` → `consumed = total`, `-1`.
- Mức 3: `vals[bisect_left(prefix, consumed)]`.

**Bài 5**

- Mức 1: tách `(chữ, số lần)` bằng regex hoặc duyệt (số có nhiều chữ số).
- Mức 2: giữ `(chỉ số cặp, đã dùng)` như RLE.
- Mức 3: `hasNext` ⇔ còn cặp với lần lặp còn lại.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
assert run_ops(RLEIterator, ["RLEIterator", "next", "next", "next"], [[[1, 4, 2, 6]], [1], [2], [1]]) == [None, 4, 6, -1]
assert run_ops(RLEIterator, ["RLEIterator", "next", "next"], [[[0, 1, 3, 2]], [3], [1]]) == [None, 2, -1]
print("Bài 1 OK:", [None, 4, 6, -1], [None, 2, -1])

Bài 1 OK: [None, 4, 6, -1] [None, 2, -1]


### Bài 2

(a) `[1, 1, 7, 7, 7]`. (b) `1`, `7`, `-1`, `-1`.

In [34]:
enc = [2, 1, 0, 5, 3, 7]
assert rle_decode(enc) == [1, 1, 7, 7, 7]
it = RLEPointers(list(enc))
states = []
for n in (1, 2, 3, 1):
    r = it.next(n)
    states.append((r, (it.i, it.used)))
print(states)
assert [s[0] for s in states] == [1, 7, -1, -1] == run_all(enc, [1, 2, 3, 1])
print("Bài 2 OK")

[(1, (0, 1)), (7, (4, 1)), (-1, (6, 0)), (-1, (6, 0))]
Bài 2 OK


### Bài 3

In [35]:
class PointersAnswer:
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]


assert run_ops(PointersAnswer, example_ops, example_args) == example_out
for enc, calls in tests[:400]:
    o = RLEOracle(list(enc))
    it = PointersAnswer(list(enc))
    assert [it.next(n) for n in calls] == [o.next(n) for n in calls]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MyIteratorAnswer:
    def __init__(self, encoding):
        self.vals = encoding[1::2]
        self.prefix = list(itertools.accumulate(encoding[0::2]))
        self.consumed = 0

    def next(self, n):
        if self.consumed + n > self.prefix[-1]:
            self.consumed = self.prefix[-1]
            return -1
        self.consumed += n
        return self.vals[bisect_left(self.prefix, self.consumed)]


assert run_ops(MyIteratorAnswer, example_ops, example_args) == example_out
for enc, calls in tests[:400]:
    o = RLEOracle(list(enc))
    it = MyIteratorAnswer(list(enc))
    assert [it.next(n) for n in calls] == [o.next(n) for n in calls]
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
import re


class StringIteratorAnswer:
    def __init__(self, compressed):
        pairs = re.findall(r"([A-Za-z])(\d+)", compressed)
        self.enc = []
        for ch, cnt in pairs:
            self.enc += [int(cnt), ch]
        self.i, self.used = 0, 0

    def hasNext(self):
        while self.i < len(self.enc) and self.enc[self.i] - self.used == 0:
            self.i, self.used = self.i + 2, 0
        return self.i < len(self.enc)

    def next(self):
        if not self.hasNext():
            return " "
        self.used += 1
        return self.enc[self.i + 1]


s = StringIteratorAnswer("L1e2t1C1o1d1e1")
assert "".join(s.next() for _ in range(8)) == "LeetCode" and s.next() == " " and not s.hasNext()
s = StringIteratorAnswer("a12b1")
assert "".join(s.next() for _ in range(13)) == "a" * 12 + "b" and s.next() == " "
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (bộ lặp trên dãy nén: trạng thái `(đoạn, đã dùng)`):**

```python
class RLEIterator:
    def __init__(self, encoding):
        self.enc, self.i, self.used = encoding, 0, 0

    def next(self, n):
        while self.i < len(self.enc) and n > self.enc[self.i] - self.used:
            n -= self.enc[self.i] - self.used
            self.used = 0
            self.i += 2
        if self.i >= len(self.enc):
            return -1
        self.used += n
        return self.enc[self.i + 1]
```

**Checklist trước khi submit:**

- [ ] Không bung dãy (độ dài tới `10^9`)?
- [ ] Điều kiện nhảy đoạn là `n > còn lại` (không `>=`)?
- [ ] Đặt `used = 0` khi sang đoạn mới; `i += 2`?
- [ ] Đoạn độ dài `0` được bỏ qua tự động?
- [ ] Không đủ `n` phần tử ⇒ `-1` (và coi như hết dãy)?
- [ ] Không dùng `or -1` (giá trị `0` hợp lệ)?
- [ ] Đã thử ví dụ `[3,8,0,9,2,5]` với `2,1,1,2`, mọi đoạn `0`, `n` đúng bằng phần còn lại?
- [ ] Đúng tên `class RLEIterator`, `next(self, n)`, không in debug.